# Card Flip Memory Game

Two teams compete to find matching pairs of cards!

## Rules
1. Take turns flipping 2 cards per turn
2. If the cards match, you score a point!
3. If they don't match, they flip back face-down
4. The team with the most matches wins!

In [1]:
from pynqsim import SimulationClient

# Connect to the Genesis server
SERVER_IP = "172.20.166.171"  # <-- Change this to your server's IP
SERVER_PORT = 9001        # <-- Change this to your server's port

sim = SimulationClient(SERVER_IP, SERVER_PORT)
print("Connected to server!")

Connected to server!


## Instructor Only: Start the Competition

Run this cell only if you are the instructor starting the game.

In [2]:
# INSTRUCTOR ONLY: Start the competition
ADMIN_PASSWORD = "admin123"  # <-- Change this

sim.admin_start_competition("competition_card_flip", password=ADMIN_PASSWORD)
print("Competition started! Teams can now join.")

Competition started! Teams can now join.


## Join the Competition

Choose your team: `team_red` or `team_blue`

In [ ]:
# Join as team_red (change to "team_blue" for the other team)
TEAM = "team_red"

sim.join_competition(team_id=TEAM)
print(f"Joined as {TEAM}!")

## Check Game State

See whose turn it is and the current scores.

In [ ]:
state = sim.get_competition_state()
print(f"Current turn: {state.get('current_turn', 'N/A')}")
print(f"Scores: {state.get('scores', {})}")

## The Grid

The cards are arranged in a 4x5 grid:

```
     Col 0   Col 1   Col 2   Col 3   Col 4
Row 3  [ ]     [ ]     [ ]     [ ]     [ ]
Row 2  [ ]     [ ]     [ ]     [ ]     [ ]
Row 1  [ ]     [ ]     [ ]     [ ]     [ ]
Row 0  [ ]     [ ]     [ ]     [ ]     [ ]
```

- **Red robot** (left side): Best for cols 0-2
- **Blue robot** (right side): Best for cols 2-4

Use `sim.flip_card(row, col)` to reveal a card!

In [ ]:
# Color names for display
COLOR_NAMES = [
    "RED", "GRN", "BLU", "YEL", "MAG",
    "CYN", "ORG", "PUR", "BRN", "PNK"
]

def show_grid(sim):
    """Display the current grid state with color names."""
    grid = sim.get_grid_state()
    num_rows = len(grid)
    num_cols = len(grid[0]) if num_rows > 0 else 0
    
    print("\n" + "="*60)
    print("CARD GRID - ? = hidden, M = matched")
    print("="*60)
    
    # Print column headers
    header = "        "
    for col_idx in range(num_cols):
        header += f"Col{col_idx}   "
    print("\n" + header)
    
    # Print rows from top to bottom
    for row_idx in range(num_rows - 1, -1, -1):
        row_str = f"Row {row_idx}  "
        for col_idx in range(num_cols):
            card = grid[row_idx][col_idx]
            if card['matched']:
                symbol = " M  "
            elif card['flipped']:
                color_name = COLOR_NAMES[card['color_idx']] if card['color_idx'] < len(COLOR_NAMES) else f"{card['color_idx']:2d}"
                symbol = f"{color_name:^4}"
            else:
                symbol = " ?  "
            row_str += f" [{symbol}]"
        print(row_str)
    print("\n" + "="*60)

show_grid(sim)

## Flip Your First Card

Pick any position to reveal a card!

In [ ]:
# Flip the first card
result1 = sim.flip_card(row=0, col=0)
print(f"Card at (0, 0) color: {result1['color_idx']}")

show_grid(sim)

## Flip Your Second Card

Try to find the matching color!

In [ ]:
# Flip the second card - try to match!
result2 = sim.flip_card(row=1, col=1)
print(f"Card at (1, 1) color: {result2['color_idx']}")

# Check if we got a match
match_result = result2.get('match_result', {})
if match_result.get('matched'):
    print("\n*** MATCH! You scored a point! ***")
else:
    print("\nNo match - cards flipping back...")

show_grid(sim)

## Check Scores

In [ ]:
state = sim.get_competition_state()
print(f"Scores: {state.get('scores', {})}")
print(f"Current turn: {state.get('current_turn', 'N/A')}")

## Helper Function: Easy Flip

Use this to quickly flip any card and see the result.

In [ ]:
def flip(sim, row, col):
    """Flip a card and show the result."""
    result = sim.flip_card(row, col)
    
    if result.get('already_flipped'):
        print(f"Card at ({row}, {col}) is already flipped!")
        return result
    
    color_idx = result['color_idx']
    color_name = COLOR_NAMES[color_idx] if color_idx < len(COLOR_NAMES) else str(color_idx)
    print(f">>> Card at ({row}, {col}) revealed: {color_name} <<<")
    
    match_result = result.get('match_result')
    if match_result:
        if match_result.get('matched'):
            print("\n*** MATCH! +1 point! ***")
        else:
            print("\nNo match. Cards hidden. Turn ended.")
        
        # Show updated scores
        state = sim.get_competition_state()
        print(f"Scores: {state.get('scores', {})}")
        print(f"Next turn: {state.get('current_turn', 'N/A')}")
    
    return result

# Example: flip card at row 2, column 3
# flip(sim, 2, 3)

## Play a Turn

Flip two cards to complete your turn!

In [ ]:
# Your turn! Flip two cards:
flip(sim, 2, 0)  # First card

In [ ]:
flip(sim, 2, 1)  # Second card

In [ ]:
show_grid(sim)

## Team Blue's Turn

Now let's simulate Team Blue joining and playing. In a real game, this would be on a separate board/notebook.

In [ ]:
# Create a second connection for Team Blue
sim_blue = SimulationClient(SERVER_IP, SERVER_PORT)

# Join as team_blue
sim_blue.join_competition(team_id="team_blue")
print("Team Blue joined!")

In [ ]:
# Check whose turn it is
state = sim_blue.get_competition_state()
print(f"Current turn: {state.get('current_turn')}")
print(f"Scores: {state.get('scores', {})}")

In [ ]:
# Team Blue flips their first card (blue robot will move!)
result1 = sim_blue.flip_card(row=2, col=4)
print(f"Team Blue flipped card at (2, 4) - color: {result1['color_idx']}")

In [ ]:
# Team Blue flips their second card
result2 = sim_blue.flip_card(row=3, col=3)
print(f"Team Blue flipped card at (3, 3) - color: {result2['color_idx']}")

match_result = result2.get('match_result', {})
if match_result.get('matched'):
    print("\n*** MATCH! Team Blue scored! ***")
else:
    print("\nNo match - cards flipping back...")

In [ ]:
# Show current grid state
show_grid(sim_blue)

In [ ]:
# Team Blue leaves
sim_blue.leave_competition()
print("Team Blue left the competition")

## Leave Competition

In [ ]:
sim.leave_competition()
print("Left the competition")

In [ ]:
sim.destroy()
print("Environment destroyed. Goodbye!")

## Instructor Only: Stop Competition

In [ ]:
# INSTRUCTOR ONLY
# sim.admin_stop_competition(password=ADMIN_PASSWORD)
# print("Competition stopped")